# TFG: Detección precoz de cáncer colorrectal utilizando técnicas de aprendizaje automático
**Grado en Física - Universidad Europea de Valencia**

* **Presentado por:** Gabriela Hortensia Ramos Mizrachi
* **Tutor:** Héctor Gisbert Mullor
* **Curso:** 2025-2026

---
### Resumen del Notebook
Este notebook documenta el proceso experimental en el cual se busca el modelo mas optimo para los datos.

# Etapa 2: Optimización mediante aprendizaje automático: 
## Mejor modelo

### Importación de Librerías y Dependencias
En esta sección se cargan todas las herramientas necesarias para el proyecto:

In [1]:
import os
import re
import shap
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, 
                             classification_report, roc_auc_score)

warnings.filterwarnings('ignore')

# Modelos
import sklearn.svm as SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Preprocesamiento
En esta sección se transforma la información cruda de los archivos Excel en una matriz de datos apta para Machine Learning:
* **Umbral de Abundancia**: Se filtran los compuestos con una presencia ≥ 5% para eliminar ruido analítico.
* **Vector de Características**: Se identifican los 100 compuestos más frecuentes en toda la cohorte (Sanos, Enfermos y Pólipos).
* **Matriz Binaria**: Se genera una tabla donde `1` indica presencia y `0` ausencia del compuesto en cada paciente.

In [ ]:
# Configuración de Rutas y Parámetros 
RUTA_BASE = r"C:\Users\User\OneDrive\Documentos\TFG\Gabriela"
UMBRAL = 0.05  # Solo se consideran compuestos con abundancia >= 5% 
OUTPUT_FILE = 'data_prepared.xlsx'
GRUPOS = {
    "M": {"carpeta": "ENFERMOS", "prefijo": "M", "n": 58},
    "S": {"carpeta": "SANOS", "prefijo": "S", "n": 48},
    "P": {"carpeta": "POLIPOS", "prefijo": "P", "n": 41}
}

def obtener_nombres_validos(ruta, umbral):
    """Extrae nombres únicos que superan el umbral de abundancia."""
    try:
        df = pd.read_excel(ruta, header=None)
        
        # Localizar el inicio de los datos (buscando 'Library/ID' o '1=')
        mask = df.apply(lambda row: row.astype(str).str.contains('Library/ID', case=False, na=False)).any(axis=1)
        if not mask.any(): return set()
        idx_inicio = mask.idxmax()
        
        data = df.iloc[idx_inicio + 1:].copy()
        
        # Columna 3: Abundancia, Columna 4: Nombre del compuesto
        data[3] = pd.to_numeric(data[3], errors='coerce')
        
        # Filtrar por abundancia y limpiar nombres
        filtro = (data[3] >= umbral) & (data[4].notna())
        nombres = data.loc[filtro, 4].astype(str).apply(lambda x: x.split('$$')[0].strip())
        
        return set(nombres)
    except Exception:
        return set()

# Procesamiento de todas las poblaciones
conteo_global = pd.DataFrame()

for etiqueta, info in GRUPOS.items():
    dir_path = os.path.join(RUTA_BASE, info["carpeta"])
    
    for i in range(1, info["n"] + 1):
        file_path = os.path.join(dir_path, f"{info['prefijo']} ({i}).xlsx")
        
        if os.path.exists(file_path):
            nombres_muestra = obtener_nombres_validos(file_path, UMBRAL)
            # Crear DataFrame temporal para sumar al conteo
            temp_df = pd.DataFrame({'Name': list(nombres_muestra), 'Count': 1})
            conteo_global = pd.concat([conteo_global, temp_df], ignore_index=True)

# Generación del Vector de Características
# Agrupamos por nombre y sumamos cuántas veces aparece en TOTAL (M + S + P)
top_names_series = conteo_global.groupby('Name')['Count'].sum()

# Ordenar por frecuencia y tomar los 100 más comunes
top_names = top_names_series.sort_values(ascending=False).head(100)
top_names_vector = top_names.index.to_list()


### Preprocesamiento y Selección de Atributos
Antes del entrenamiento, los datos se someten a un pipeline de refinamiento:
-  **Limpieza de Varianza**: Eliminación de columnas con información constante.
- **Escalado**: Normalización de datos mediante `StandardScaler`.
-  **Reducción de Dimensionalidad**: Selección de los 25 mejores predictores (SelectKBest) para prevenir el sobreajuste (overfitting).
- **Validación**: División del dataset en entrenamiento (75%) y test (25%) con estratificación para mantener el balance de clases.

In [ ]:
def procesar_nombres_muestra(ruta, umbral):
    """Extrae y limpia nombres de compuestos que superan el umbral."""
    try:
        df = pd.read_excel(ruta, header=None)
        # Localizar el inicio de los datos
        mask = df.apply(lambda row: row.astype(str).str.contains('Library/ID', case=False, na=False)).any(axis=1)
        if not mask.any(): return set()
        idx_inicio = mask.idxmax()
        
        data = df.iloc[idx_inicio + 1:].copy()
        data[3] = pd.to_numeric(data[3], errors='coerce') # Columna de abundancia
        
        # Filtrar y limpiar (quitar $$ y espacios)
        filtro = (data[3] >= umbral) & (data[4].notna())
        nombres = data.loc[filtro, 4].astype(str).apply(lambda x: x.split('$$')[0].strip())
        return set(nombres)
    except Exception:
        return set()

# Obtener el Vector de Características 
conteo_global = []

for etiqueta, info in GRUPOS.items():
    dir_path = os.path.join(RUTA_BASE, info["carpeta"])
    for i in range(1, info["n"] + 1):
        file_path = os.path.join(dir_path, f"{info['prefijo']} ({i}).xlsx")
        if os.path.exists(file_path):
            nombres = procesar_nombres_muestra(file_path, UMBRAL)
            conteo_global.extend(list(nombres))

# Calculamos los 100 más repetidos
top_names_vector = pd.Series(conteo_global).value_counts().head(100).index.tolist()

# Construir la Matriz Binaria (data_prepared.xlsx)
rows = []

for etiqueta, info in GRUPOS.items():
    dir_path = os.path.join(RUTA_BASE, info["carpeta"])
    for i in range(1, info["n"] + 1):
        file_name = f"{info['prefijo']} ({i}).xlsx"
        file_path = os.path.join(dir_path, file_name)
        
        if os.path.exists(file_path):
            # Obtener nombres presentes en esta muestra específica
            nombres_presentes = procesar_nombres_muestra(file_path, UMBRAL)
            
            # Crear diccionario de presencia (1 o 0)
            # Usamos "in" porque ya limpiamos los nombres en el paso anterior
            presencia = {name: (1 if name in nombres_presentes else 0) for name in top_names_vector}
            
            # Añadir metadatos
            row = {
                'File': file_name,
                'State': etiqueta,
                **presencia
            }
            rows.append(row)

# Crear DataFrame final
summary_df = pd.DataFrame(rows)

# Guardar Resultados
summary_df.to_excel(OUTPUT_FILE, index=False)

### Evaluación Comparativa de Algoritmos
Se realiza una búsqueda de hiperparámetros mediante `GridSearchCV` para 8 algoritmos distintos. 

In [ ]:
# Carga y preparación de datos para modelado

df = pd.read_excel('data_prepared.xlsx')
df['target'] = df['State'].apply(lambda x: 0 if x == 'S' else 1)
X_raw = df.drop(columns=['File', 'State', 'target'], errors='ignore')
y = df['target']

# Preprocesamiento
X_red = VarianceThreshold(threshold=0.0).fit_transform(X_raw)
X_scaled = StandardScaler().fit_transform(X_red)
X_final = SelectKBest(f_classif, k=min(25, X_scaled.shape[1])).fit_transform(X_scaled, y)

# Convertir a DataFrame 
cols = [f"Compuesto_{i}" for i in range(X_final.shape[1])]
X_train, X_test, y_train, y_test = train_test_split(
    pd.DataFrame(X_final, columns=cols), y, test_size=0.25, random_state=42, stratify=y
)

# Definiciion de hiperparametros

configs = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=10000, solver='saga'),
        'params': {
            'C': np.logspace(-3, 2, 6).tolist(),
            'penalty': ['l1', 'l2']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': np.linspace(3, 20, 5, dtype=int).tolist() + [None],
            'min_samples_leaf': np.linspace(1, 10, 4, dtype=int).tolist()
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': np.linspace(100, 1000, 4, dtype=int).tolist(),
            'max_depth': [None, 10, 20, 30]
        }
    },
    'SVM': {
        'model': SVC(probability=True, random_state=42),
        'params': {
            'C': np.logspace(-2, 2, 5).tolist(), # 5 pasos logarítmicos
            'gamma': ['scale', 'auto'] + np.logspace(-3, -1, 3).tolist(),
            'kernel': ['rbf', 'linear']
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': np.linspace(3, 25, 6, dtype=int).tolist(),
            'weights': ['uniform', 'distance']
        }
    },
    'XGBoost': {
        'model': XGBClassifier(eval_metric='logloss', random_state=42),
        'params': {
            'learning_rate': np.linspace(0.01, 0.3, 5).tolist(), 
            'n_estimators': [100, 300, 500],
            'max_depth': [3, 5, 7]
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(verbose=-1, random_state=42),
        'params': {
            'learning_rate': np.linspace(0.01, 0.2, 5).tolist(),
            'num_leaves': np.linspace(15, 63, 4, dtype=int).tolist(),
            'n_estimators': [100, 300]
        }
    },
    'CatBoost': {
        'model': CatBoostClassifier(verbose=0, random_state=42),
        'params': {
            'learning_rate': [0.05],
            'depth': [4, 6, 8],
            'iterations': [100, 200]
        }
    }
}

# Ejecución de GridSearchCV para cada modelo y recopilación de resultados
resultados = []
print(f"\n{'ALGORITMO':<20} | {'ACC':<6} | {'PREC':<6} | {'REC':<6} | {'F1':<6} | {'AUC':<6} | {'TIEMPO'} | {'MEJORES PARÁMETROS'}")
print("-" * 125)

for nombre, config in configs.items():
    grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='f1', n_jobs=-1)
    
    t_inicio = time.time()
    grid.fit(X_train, y_train)
    mejor_modelo = grid.best_estimator_
    
    y_pred = mejor_modelo.predict(X_test)
    y_proba = mejor_modelo.predict_proba(X_test)[:, 1]
    t_fin = time.time()
    
    # Métricas
    acc, prec, rec, f1 = accuracy_score(y_test, y_pred), precision_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_proba)
    duracion = t_fin - t_inicio
    
    resultados.append({
        'Modelo': nombre, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 
        'F1-Score': f1, 'AUC-ROC': auc_val, 'Tiempo': duracion, 'Params': grid.best_params_
    })
    
    print(f"{nombre:<20} | {acc:.3f} | {prec:.3f} | {rec:.3f} | {f1:.3f} | {auc_val:.3f} | {duracion:>6.2f}s | {grid.best_params_}")

# Guardar resultados finales
df_final = pd.DataFrame(resultados).sort_values(by='AUC-ROC', ascending=False)


ALGORITMO            | ACC    | PREC   | REC    | F1     | AUC    | TIEMPO | MEJORES PARÁMETROS
-----------------------------------------------------------------------------------------------------------------------------
Logistic Regression  | 0.676 | 0.676 | 1.000 | 0.806 | 0.838 |   0.23s | {'C': 0.01, 'penalty': 'l2'}
Decision Tree        | 0.595 | 0.679 | 0.760 | 0.717 | 0.550 |   0.33s | {'max_depth': 3, 'min_samples_leaf': 7}
Random Forest        | 0.838 | 0.828 | 0.960 | 0.889 | 0.888 |   8.68s | {'max_depth': 10, 'n_estimators': 100}
SVM                  | 0.676 | 0.676 | 1.000 | 0.806 | 0.748 |   1.28s | {'C': 0.01, 'gamma': 'scale', 'kernel': 'linear'}
KNN                  | 0.676 | 0.783 | 0.720 | 0.750 | 0.750 |   0.17s | {'n_neighbors': 20, 'weights': 'uniform'}
XGBoost              | 0.676 | 0.676 | 1.000 | 0.806 | 0.723 |   2.33s | {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
LightGBM             | 0.730 | 0.826 | 0.760 | 0.792 | 0.760 |  88.53s | {'lea